# Multi-sample cell typing

# Dependencies and preparation


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import anndata as ad
import scripts
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.spatial_plot import spatial_celltype_plot
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE
from scripts.anndata_helpers import load_and_build_anndata, save_h5ad

In [ ]:
import scripts
import scripts.spatial_plot
import importlib

importlib.reload(scripts.spatial_plot)

Define sample base names:

In [ ]:
samples = [
"IHOPE14_MedLN_BottomLeft",
"IHOPE14_MedLN_BottomRight",
"IHOPE14_MedLN_TopRight",
"IHOPE14_mesLN", #this one may also have _RAW_RESULTS in name
"IHOPE20_LN",
"IHOPE20_Spleen",
"IHOPE26_LN",
"IHOPE26_Spleen",
"IHOPE27_LN",
"IHOPE27_Spleen",
"IHOPE39_LN",
"IHOPE39_MesLN_1",
]

In [ ]:
samples = [
"IHOPE39_Spleen"
]

# Go

In [ ]:
from scripts.spatial_plot import spatial_celltype_plot

for basename in samples:

    try:
        print(f"Processing {basename}")

        input_path = (
            f"../data/processed/anndata/"
            f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
        )

        output_path = (
            f"../data/processed/anndata/"
            f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_updated_banksy.h5ad"
        )

        # Load data
        adata = ad.read_h5ad(input_path)
        print(f"Loaded {adata.n_obs} cells")

        # Run cell typing
        adata = assign_cell_types_bool_IHOPE(adata)

        # Type plots
        celltype_cols = [
            c for c in adata.obs.columns
            if c.startswith("type_")
            and not c.endswith("unclassified")
            and not c.endswith("Endothelial")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols,
            min_cells=50,
            title = f"{basename}"
        )

        # T subtype plots
        t_subtypes = [
            c for c in adata.obs.columns
            if c.startswith("subtype_T")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols=t_subtypes,
            min_cells=15,
            size=10,
            alpha=0.7,
            title = f"{basename}"
        )

        # B subtype plots
        b_subtypes = [
            c for c in adata.obs.columns
            if c.startswith("subtype_B_")
        ]

        spatial_celltype_plot(
            adata,
            celltype_cols=b_subtypes,
            min_cells=15,
            size=10,
            alpha=0.7,
            title = f"{basename}"
        )

        # Summary table
        df_summary = summarize_celltypes_IHOPE(
            adata,
            filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary.csv"
        )

        # Save anndata
        save_h5ad(adata, output_path)

        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error in {basename}: {e}")


# Update below

Imports and preparation

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

# Cell typing & spatial plotting scripts
from scripts.celltype_rules_IHOPE import assign_cell_types_bool_IHOPE
from scripts.summary_celltypes_IHOPE import summarize_celltypes_IHOPE
from scripts.spatial_plot import spatial_celltype_plot

# B-cell domain analysis
from scripts.banksy_domains import (
    compute_domain_bcell_stats,
    plot_domains_by_bcell_fraction,
    assign_bcell_follicles,
    plot_bcell_follicles
)

In [ ]:
import scripts
import scripts.banksy_domains
import importlib

importlib.reload(scripts.banksy_domains)

Sample names and paths

In [ ]:
samples = [
    "IHOPE14_MedLN_BottomLeft",
    "IHOPE14_Spleen",
    "IHOPE20_LN",
    "IHOPE26_LN",
    "IHOPE26_Spleen",
    "IHOPE27_LN",
    "IHOPE27_Spleen",
    "IHOPE39_LN",
    "IHOPE39_MesLN_1",
    "IHOPE39_MesLN",
    "IHOPE39_Spleen"
]

input_dir = "../data/processed/anndata"
output_dir = "../data/processed/anndata"
os.makedirs(output_dir, exist_ok=True)


Loop over samples to apply cell typing

In [ ]:
adatas = {}

for basename in samples:
    try:
        print(f"Processing {basename}")


        # Try cf5 first
        input_path = os.path.join(
        input_dir,
        f"{basename}_filtered_arcsinh_cf5_GMM_IHOPE_celltypes_banksy.h5ad"
)

        # If not found, try cf5.0
        if not os.path.exists(input_path):
            input_path = os.path.join(
                input_dir,
                f"{basename}_filtered_arcsinh_cf5.0_GMM_IHOPE_celltypes_banksy.h5ad"
            )

        if not os.path.exists(input_path):
            raise FileNotFoundError(f"No cf5 or cf5.0 file found for {basename}")

        # Load AnnData
        adata = ad.read_h5ad(input_path)
        print(f"Loaded {adata.n_obs} cells")

        # Assign / update cell types
        adata = assign_cell_types_bool_IHOPE(adata)

        # Store for later grid plotting
        adatas[basename] = adata

        # Save updated AnnData
        adata.write_h5ad(output_path)
        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error in {basename}: {e}")


Visualize cell types (type)

In [ ]:
fig, axes = plt.subplots(
    nrows=len(adatas)//3 + 1, ncols=3, figsize=(18, 6 * ((len(adatas)//3)+1))
)
axes = axes.flatten()

for ax, (basename, adata) in zip(axes, adatas.items()):
    celltype_cols = [
        c for c in adata.obs.columns
        if c.startswith("type_") and not c.endswith(("unclassified", "Endothelial"))
    ]

    # Use the plotting function
    spatial_celltype_plot(
        adata,
        celltype_cols,
        min_cells=50,
        title=basename
    )

plt.tight_layout()
plt.show()

#TODO repeat for intermediate (intermediate_*) or subtype (subtype_*) levels
#TODO fix the grid or remove all of it

Compute & plot B cell domains

In [ ]:

for basename, adata in adatas.items():
    stats_df = compute_domain_bcell_stats(adata)

    plot_domains_by_bcell_fraction(
    adata,
    stats_df,
    sample_name=basename
)


Select B cell follicle domains

In [ ]:
sample_name = "IHOPE14_MedLN_BottomLeft"   # choose sample
follicle_domains = [8]                 # domains you picked from the plot

adata = adatas[sample_name]

adata = assign_bcell_follicles(
    adata,
    follicle_domains=follicle_domains
)

plot_bcell_follicles(
    adata,
    sample_name=sample_name
)

# store back into dict
adatas[sample_name] = adata


In [ ]:
sample_name = "IHOPE39_Spleen"   # choose sample
follicle_domains = [6]                 # domains you picked from the plot

adata = adatas[sample_name]

adata = assign_bcell_follicles(
    adata,
    follicle_domains=follicle_domains
)

plot_bcell_follicles(
    adata,
    sample_name=sample_name
)

# store back into dict
adatas[sample_name] = adata


Placeholder for Spatially Constrained GC B / Plasmablast / TfH Assignment

In [ ]:
for basename, adata in adatas.items():
    # Placeholder: here you will spatially prune subtypes
    # Example:
    # adata = spatially_define_celltype(
    #     adata,
    #     celltype_key="subtype_B_GC",
    #     domain_key="B_follicle",
    #     inside=True  # True = keep only inside follicle
    # )

    # adata = spatially_define_celltype(
    #     adata,
    #     celltype_key="subtype_B_plasmablast",
    #     domain_key="B_follicle",
    #     inside=False  # True = keep only inside follicle
    # )

    print(f"Spatial pruning placeholder for {basename}")


In [ ]:
for basename, adata in adatas.items():
    df_summary = summarize_celltypes_IHOPE(
        adata,
        filename=f"{basename}_filtered_arcsinh_cf5.0_IHOPE_summary_updated.csv"
    )


Updated summary tables